# Multi-Goal Financial Asset Recommender System - MVP

- **[TEAM MEMBER A] Data Engineering & PyTorch ML**: Web Scraping, Dynamic DataFrame Preprocessing (One-Hot & NaN filling natively to CSV), Masked Weighted PyTorch Autoencoder.
- **[TEAM MEMBER B] Scoring & Allocation**: Similarity Score, Dynamic K Filter, Softmax.
- **[TEAM MEMBER C] SORR Simulation**: Evaluation loops, Path-Dependent withdrawals, GFR, ETV.
- **[UNIFIED DASHBOARD]**: Centralized Config and Hyperparameter Search.


In [ ]:
!pip install yfinance matplotlib seaborn scipy lxml html5lib requests tqdm nbformat torch
import sqlite3
import pandas as pd
import numpy as np
import io
import math, time, os, requests
from scipy.stats import norm
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

plt.style.use('default')
%matplotlib inline

## [TEAM MEMBER A] Layer 1: Data Scraping & PyTorch Embeddings

In [ ]:
def fetch_sp1500_universe():
    print(f"[{time.strftime('%H:%M:%S')}] [Member A] Fetching latest S&P 1500 constituents from Wikipedia...")
    headers = {'User-Agent': 'Mozilla/5.0'}
    urls = [
        "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",
        "https://en.wikipedia.org/wiki/List_of_S%26P_400_companies",
        "https://en.wikipedia.org/wiki/List_of_S%26P_600_companies"
    ]
    tickers = []
    for u in urls:
        html = requests.get(u, headers=headers).text
        df = pd.read_html(io.StringIO(html))[0]
        tickers.extend(df['Symbol'].tolist())
    
    clean_tickers = [t.replace('.', '-') for t in set(tickers)]
    print(f"[{time.strftime('%H:%M:%S')}] [Member A] Discovered {len(clean_tickers)} symbols.")
    return clean_tickers

def generate_dataset_member_a(tickers, config):
    MASTER_FILE = 'sp1500_master_research_dataset.csv'
    PRICE_FILE = 'sp1500_price_matrix.csv'
    
    if config["data_source_mode"] == 'CSV' and os.path.exists(MASTER_FILE) and os.path.exists(PRICE_FILE):
        print(f"[{time.strftime('%H:%M:%S')}] [Member A] Loading cached Master DataFrame and Price Matrix from CSV...")
        master_df = pd.read_csv(MASTER_FILE, index_col='ticker')
        price_matrix = pd.read_csv(PRICE_FILE, index_col=0, parse_dates=True)
    else:
        if len(tickers) == 0:
            print("[Member A] WARNING: CSV not found and no tickers provided. Defaulting to SP1500 scrape.")
            tickers = fetch_sp1500_universe()
            
        print(f"[{time.strftime('%H:%M:%S')}] [Member A] Scraping bulk data from yfinance...")
        raw_data = yf.download(tickers, start=config["data_start_date"], group_by="ticker", auto_adjust=True, progress=False)
        price_matrix = raw_data.xs('Close', axis=1, level=1).ffill() if isinstance(raw_data.columns, pd.MultiIndex) else pd.DataFrame({tickers[0]: raw_data['Close']}).ffill()
        if "VBTIX" not in price_matrix.columns:
            price_matrix["VBTIX"] = np.cumprod(1 + np.random.normal(0.04/252, 0.05/np.sqrt(252), size=len(price_matrix)))
        
        print(f"[{time.strftime('%H:%M:%S')}] [Member A] Fetching Extended Fundamental Context Data...")
        company_data = []
        for ticker in tqdm(tickers):
            if ticker == "VBTIX" or ticker == "^GSPC":
                company_data.append({"ticker": ticker, "sector": "Index/Bond", "industry": "Index", "quoteType": "ETF", "top_holdings": "['"+ticker+"']"})
                continue
            try:
                t_obj = yf.Ticker(ticker)
                info = t_obj.info
                time.sleep(config["scrape_delay"])
                
                quote_type = info.get('quoteType', '')
                holdings = [ticker]
                if quote_type == 'ETF':
                    funds = getattr(t_obj, 'funds_data', None)
                    if funds and hasattr(funds, 'top_holdings') and funds.top_holdings is not None:
                        holdings = list(funds.top_holdings.index)
                
                # Copy everything to ensure massive dataset options
                company_record = info.copy()
                company_record['ticker'] = ticker
                company_record['quoteType'] = quote_type
                company_record['top_holdings'] = str(holdings)
                
                if 'sector' not in company_record: company_record['sector'] = 'Unknown'
                if 'industry' not in company_record: company_record['industry'] = 'Unknown'
                
                company_data.append(company_record)
            except Exception as e:
                pass
                
        master_df = pd.DataFrame(company_data).set_index('ticker')
        master_df.to_csv(MASTER_FILE)
        price_matrix.to_csv(PRICE_FILE)

    daily_returns = price_matrix.pct_change().dropna(how='all')
    
    # =========================================================================
    # ML FEATURE PREPROCESSING GATE
    # Maps directly onto master_df so preprocessing doesn't clutter ML Engine
    # =========================================================================
    print(f"\n[{time.strftime('%H:%M:%S')}] [Member A] Preprocessing Configured Native DataFrame ML Features...")
    feature_config = config.get("ml_training_features", [])
    
    for f in feature_config:
        if f in ['hist_momentum', 'hist_volatility']:
            continue
            
        if f in master_df.columns:
            if master_df[f].dtype == 'object' or pd.api.types.is_categorical_dtype(master_df[f]):
                # Dynamically One-Hot Encode Categoricals onto master_df natively
                print(f"   -> Extracting Categorical Matrix: '{f}'")
                # Drop original string column after dummy map to preserve CSV shape properly
                dummies = pd.get_dummies(master_df[f], prefix=f, dtype=float)
                master_df = pd.concat([master_df, dummies], axis=1)
                master_df.drop(columns=[f], inplace=True)
            else:
                # Fill Numerics dynamically to avoid Tensor Exceptions
                print(f"   -> Processing Numeric Vector: '{f}'")
                master_df[f] = pd.to_numeric(master_df[f], errors='coerce').fillna(0.0)
        else:
            print(f"   -> WARNING: Requested config feature '{f}' not found in master_df.")

    # Re-save the master CSV with the preprocessed columns attached securely
    if config["data_source_mode"] != 'CSV':
        master_df.to_csv(MASTER_FILE)

    print("\n" + "="*50)
    print("=== DATAFRAME DIAGNOSTICS & SYSTEM MAPPING ===")
    print("="*50)
    print(f"Total Equities Tracked: {len(master_df)}")
    print(f"Total Columns Processed per Asset: {len(master_df.columns)}\n")
    print(f"All Features Extracted in Master DF:\n{list(master_df.columns)}\n")
    
    print("[PIPELINE DATA FLOW MAPPING]")
    print(f"-> Selected for ML Tensor Construction: {config['ml_training_features']}")
    print("-> Passed to Member B (Allocation Layer 2): ['current_volatility'], dynamic_embeddings")
    print("-> Passed to Member C (Simulation Layer 3): daily_returns, price_matrix")
    print("-> Ignored by Logic / For Display Only: ALL OTHER COLUMNS (Data Poisoning Shield Active)")
    print("==============================================\n")
    
    return master_df, price_matrix, daily_returns


In [ ]:
# =========================================================================
# DATA DICTIONARY & VALIDATION ENGINE
# =========================================================================
# Classifies every column by data-poisoning risk for backtesting embeddings.
#
# Safety Classes:
#   SAFE_STATIC    — Time-invariant company identity (sector, country).
#                    These rarely change and are safe to use across all backtest years.
#   SLOW_CHANGING  — Changes gradually over years (employees, governance scores).
#                    Low risk but values used are TODAY's snapshot, not historical.
#   POINT_IN_TIME  — Current market/financial snapshot (price, PE, marketCap).
#                    HIGH RISK: using today's value to predict historical returns = data leakage.
#   DERIVED        — Computed per-year inside the training loop (hist_momentum, hist_volatility).
#                    Properly time-aligned. No leakage.
#   METADATA       — API/system plumbing (timestamps, IDs, booleans). Not useful for ML.
#   TEXT           — Free-text fields requiring NLP. Cannot be one-hot encoded.
#   IDENTIFIER     — Names, symbols, phone numbers, URLs. Not features.
# =========================================================================

COLUMN_SAFETY_REGISTRY = {
    # ── Company Identity (SAFE_STATIC) ──
    "sector":       ("GICS sector classification (e.g. Technology, Healthcare). ~12 unique values.", "SAFE_STATIC"),
    "sectorKey":    ("Sector key identifier (lowercase slug of sector).", "SAFE_STATIC"),
    "sectorDisp":   ("Sector display name (same as sector, alternate key).", "SAFE_STATIC"),
    "industry":     ("GICS industry sub-classification (e.g. Semiconductors). ~150 unique values.", "SAFE_STATIC"),
    "industryKey":  ("Industry key identifier (lowercase slug).", "SAFE_STATIC"),
    "industryDisp": ("Industry display name (alternate key).", "SAFE_STATIC"),
    "industrySymbol": ("Industry symbol code.", "SAFE_STATIC"),
    "quoteType":    ("Security type: EQUITY, ETF, MUTUALFUND. Defines asset class.", "SAFE_STATIC"),
    "state":        ("US state or province of company headquarters. ~50 unique values.", "SAFE_STATIC"),
    "country":      ("Country of domicile. Mostly 'United States'. ~5 unique values.", "SAFE_STATIC"),
    "city":         ("City of headquarters. ⚠️ HIGH cardinality (~800+ unique). One-hot will explode tensor width.", "SAFE_STATIC"),
    "address1":     ("Street address of HQ. ⚠️ EXTREME cardinality. Not useful for embeddings.", "IDENTIFIER"),
    "address2":     ("Street address line 2.", "IDENTIFIER"),
    "zip":          ("Postal/zip code. ⚠️ HIGH cardinality (~1000+ unique).", "SAFE_STATIC"),
    "exchange":     ("Exchange code (NMS, NYQ, etc.). ~5 unique values.", "SAFE_STATIC"),
    "fullExchangeName": ("Full exchange name (NASDAQ, NYSE, etc.).", "SAFE_STATIC"),
    "market":       ("Market identifier (e.g. us_market). Almost always the same.", "SAFE_STATIC"),
    "lastSplitFactor": ("Last stock split ratio (e.g. '4:1'). Historical event.", "SAFE_STATIC"),

    # ── Slowly Changing Fundamentals (SLOW_CHANGING) ──
    "fullTimeEmployees":    ("Number of full-time employees. Changes annually. Today's snapshot.", "SLOW_CHANGING"),
    "auditRisk":            ("ISS audit committee risk score (1-10, 10=highest). Updated ~annually.", "SLOW_CHANGING"),
    "boardRisk":            ("ISS board composition risk score (1-10). Updated ~annually.", "SLOW_CHANGING"),
    "compensationRisk":     ("ISS executive compensation risk score (1-10). Updated ~annually.", "SLOW_CHANGING"),
    "shareHolderRightsRisk": ("ISS shareholder rights risk score (1-10). Updated ~annually.", "SLOW_CHANGING"),
    "overallRisk":          ("ISS composite governance risk score (1-10). Updated ~annually.", "SLOW_CHANGING"),

    # ── Price-Derived & Market Snapshot (POINT_IN_TIME) ──
    # ⚠️ ALL of these use TODAY's price/financials. Using them in a backtest
    #    that loops through historical years causes DATA LEAKAGE because the
    #    model sees 2024 values when 'predicting' 2015 forward returns.
    "marketCap":                ("Current market cap (price × shares). ⚠️ CHANGES DAILY with stock price.", "POINT_IN_TIME"),
    "nonDilutedMarketCap":      ("Market cap excluding dilutive securities. ⚠️ Price-derived.", "POINT_IN_TIME"),
    "enterpriseValue":          ("Market cap + debt - cash. ⚠️ Price-derived.", "POINT_IN_TIME"),
    "currentPrice":             ("Current trading price.", "POINT_IN_TIME"),
    "previousClose":            ("Yesterday's closing price.", "POINT_IN_TIME"),
    "open":                     ("Today's opening price.", "POINT_IN_TIME"),
    "dayLow":                   ("Today's intraday low.", "POINT_IN_TIME"),
    "dayHigh":                  ("Today's intraday high.", "POINT_IN_TIME"),
    "regularMarketPreviousClose": ("Previous close (regular session). Duplicate of previousClose.", "POINT_IN_TIME"),
    "regularMarketOpen":        ("Opening price (regular session).", "POINT_IN_TIME"),
    "regularMarketDayLow":      ("Day low (regular session).", "POINT_IN_TIME"),
    "regularMarketDayHigh":     ("Day high (regular session).", "POINT_IN_TIME"),
    "regularMarketPrice":       ("Current regular market price.", "POINT_IN_TIME"),
    "regularMarketVolume":      ("Today's trading volume.", "POINT_IN_TIME"),
    "regularMarketChange":      ("Today's absolute price change.", "POINT_IN_TIME"),
    "regularMarketChangePercent": ("Today's price change %.", "POINT_IN_TIME"),
    "regularMarketDayRange":    ("Today's price range as string.", "POINT_IN_TIME"),
    "postMarketPrice":          ("After-hours trading price.", "POINT_IN_TIME"),
    "postMarketChange":         ("After-hours price change.", "POINT_IN_TIME"),
    "postMarketChangePercent":  ("After-hours price change %.", "POINT_IN_TIME"),
    "bid":                      ("Current best bid price.", "POINT_IN_TIME"),
    "ask":                      ("Current best ask price.", "POINT_IN_TIME"),
    "bidSize":                  ("Lot size at best bid.", "POINT_IN_TIME"),
    "askSize":                  ("Lot size at best ask.", "POINT_IN_TIME"),
    "volume":                   ("Today's trading volume.", "POINT_IN_TIME"),

    # ── Valuation Ratios (POINT_IN_TIME — numerator or denominator is current price) ──
    "beta":                         ("5-year monthly beta vs S&P 500. Computed from recent 5yr window.", "POINT_IN_TIME"),
    "trailingPE":                   ("Price / trailing 12-mo EPS. ⚠️ Numerator = current price.", "POINT_IN_TIME"),
    "forwardPE":                    ("Price / forward EPS estimate. ⚠️ Numerator = current price.", "POINT_IN_TIME"),
    "priceToBook":                  ("Price / book value per share. ⚠️ Numerator = current price.", "POINT_IN_TIME"),
    "priceToSalesTrailing12Months": ("Price / trailing 12-mo revenue/share. ⚠️ Price-derived.", "POINT_IN_TIME"),
    "enterpriseToRevenue":          ("EV / revenue. ⚠️ EV is price-derived.", "POINT_IN_TIME"),
    "enterpriseToEbitda":           ("EV / EBITDA. ⚠️ EV is price-derived.", "POINT_IN_TIME"),
    "trailingPegRatio":             ("PEG ratio (PE / growth). ⚠️ PE is price-derived.", "POINT_IN_TIME"),
    "priceEpsCurrentYear":          ("Price / current-year EPS estimate.", "POINT_IN_TIME"),

    # ── Dividend Metrics (POINT_IN_TIME) ──
    "dividendRate":                 ("Current annualized dividend ($/share).", "POINT_IN_TIME"),
    "dividendYield":                ("Annual dividend / current price. ⚠️ Price-derived.", "POINT_IN_TIME"),
    "payoutRatio":                  ("Dividends paid / net income. Last fiscal year snapshot.", "POINT_IN_TIME"),
    "fiveYearAvgDividendYield":     ("5-year average dividend yield.", "POINT_IN_TIME"),
    "trailingAnnualDividendRate":   ("Trailing 12-mo dividend ($/share).", "POINT_IN_TIME"),
    "trailingAnnualDividendYield":  ("Trailing dividend / current price. ⚠️ Price-derived.", "POINT_IN_TIME"),
    "lastDividendValue":            ("Most recent dividend payment amount.", "POINT_IN_TIME"),

    # ── Volume & Liquidity (POINT_IN_TIME) ──
    "averageVolume":            ("Average daily volume (3-month lookback).", "POINT_IN_TIME"),
    "averageVolume10days":      ("Average daily volume (10-day lookback).", "POINT_IN_TIME"),
    "averageDailyVolume10Day":  ("Average daily volume 10-day (duplicate key).", "POINT_IN_TIME"),
    "averageDailyVolume3Month": ("Average daily volume 3-month (duplicate key).", "POINT_IN_TIME"),

    # ── Technical Price Levels (POINT_IN_TIME) ──
    "fiftyDayAverage":                  ("50-day simple moving average price.", "POINT_IN_TIME"),
    "twoHundredDayAverage":             ("200-day simple moving average price.", "POINT_IN_TIME"),
    "fiftyDayAverageChange":            ("Absolute change from 50-day MA.", "POINT_IN_TIME"),
    "fiftyDayAverageChangePercent":     ("% change from 50-day MA.", "POINT_IN_TIME"),
    "twoHundredDayAverageChange":       ("Absolute change from 200-day MA.", "POINT_IN_TIME"),
    "twoHundredDayAverageChangePercent":("% change from 200-day MA.", "POINT_IN_TIME"),
    "fiftyTwoWeekLow":                  ("52-week low price.", "POINT_IN_TIME"),
    "fiftyTwoWeekHigh":                 ("52-week high price.", "POINT_IN_TIME"),
    "fiftyTwoWeekLowChange":            ("Price change from 52-week low.", "POINT_IN_TIME"),
    "fiftyTwoWeekLowChangePercent":     ("%% change from 52-week low.", "POINT_IN_TIME"),
    "fiftyTwoWeekHighChange":           ("Price change from 52-week high.", "POINT_IN_TIME"),
    "fiftyTwoWeekHighChangePercent":    ("% change from 52-week high.", "POINT_IN_TIME"),
    "fiftyTwoWeekChangePercent":        ("52-week price return %.", "POINT_IN_TIME"),
    "fiftyTwoWeekRange":                ("52-week price range as string.", "POINT_IN_TIME"),
    "52WeekChange":                     ("52-week price change ratio.", "POINT_IN_TIME"),
    "SandP52WeekChange":                ("S&P 500 52-week change (benchmark comparison).", "POINT_IN_TIME"),
    "allTimeHigh":                      ("All-time high price.", "POINT_IN_TIME"),
    "allTimeLow":                       ("All-time low price.", "POINT_IN_TIME"),

    # ── Income Statement & Margins (POINT_IN_TIME — last reported quarter/year) ──
    "profitMargins":        ("Net profit margin. Last reported quarter snapshot.", "POINT_IN_TIME"),
    "grossMargins":         ("Gross margin. Last reported quarter snapshot.", "POINT_IN_TIME"),
    "operatingMargins":     ("Operating margin. Last reported quarter snapshot.", "POINT_IN_TIME"),
    "ebitdaMargins":        ("EBITDA margin. Last reported quarter snapshot.", "POINT_IN_TIME"),
    "returnOnAssets":       ("Return on assets. Last reported snapshot.", "POINT_IN_TIME"),
    "returnOnEquity":       ("Return on equity. Last reported snapshot.", "POINT_IN_TIME"),
    "revenueGrowth":        ("Quarter-over-quarter revenue growth rate.", "POINT_IN_TIME"),
    "earningsGrowth":       ("Quarter-over-quarter earnings growth rate.", "POINT_IN_TIME"),
    "earningsQuarterlyGrowth": ("Quarterly earnings growth (duplicate key).", "POINT_IN_TIME"),
    "totalRevenue":         ("Total revenue in absolute $ (last reported).", "POINT_IN_TIME"),
    "revenuePerShare":      ("Revenue per share (last reported).", "POINT_IN_TIME"),
    "grossProfits":         ("Gross profit in absolute $ (last reported).", "POINT_IN_TIME"),
    "ebitda":               ("EBITDA in absolute $ (last reported).", "POINT_IN_TIME"),
    "netIncomeToCommon":    ("Net income to common shareholders in $ (last reported).", "POINT_IN_TIME"),

    # ── Balance Sheet (POINT_IN_TIME) ──
    "totalCash":            ("Total cash on hand in $.", "POINT_IN_TIME"),
    "totalCashPerShare":    ("Cash per share.", "POINT_IN_TIME"),
    "totalDebt":            ("Total debt in $.", "POINT_IN_TIME"),
    "debtToEquity":         ("Debt-to-equity ratio.", "POINT_IN_TIME"),
    "currentRatio":         ("Current assets / current liabilities.", "POINT_IN_TIME"),
    "quickRatio":           ("(Current assets - inventory) / current liabilities.", "POINT_IN_TIME"),
    "bookValue":            ("Book value per share.", "POINT_IN_TIME"),
    "freeCashflow":         ("Free cash flow in $.", "POINT_IN_TIME"),
    "operatingCashflow":    ("Operating cash flow in $.", "POINT_IN_TIME"),

    # ── EPS (POINT_IN_TIME) ──
    "trailingEps":              ("Trailing 12-month EPS.", "POINT_IN_TIME"),
    "forwardEps":               ("Forward EPS analyst estimate.", "POINT_IN_TIME"),
    "epsTrailingTwelveMonths":  ("EPS trailing 12 months (duplicate key).", "POINT_IN_TIME"),
    "epsForward":               ("Forward EPS (duplicate key).", "POINT_IN_TIME"),
    "epsCurrentYear":           ("Current fiscal year EPS estimate.", "POINT_IN_TIME"),

    # ── Ownership & Short Interest (POINT_IN_TIME) ──
    "floatShares":              ("Shares available for public trading.", "POINT_IN_TIME"),
    "sharesOutstanding":        ("Total shares outstanding.", "POINT_IN_TIME"),
    "impliedSharesOutstanding": ("Implied shares outstanding.", "POINT_IN_TIME"),
    "sharesShort":              ("Shares currently sold short.", "POINT_IN_TIME"),
    "sharesShortPriorMonth":    ("Shares short prior month.", "POINT_IN_TIME"),
    "sharesPercentSharesOut":   ("Short shares as % of outstanding.", "POINT_IN_TIME"),
    "shortRatio":               ("Days to cover short positions.", "POINT_IN_TIME"),
    "shortPercentOfFloat":      ("Short interest as % of float.", "POINT_IN_TIME"),
    "heldPercentInsiders":      ("% of shares held by insiders.", "POINT_IN_TIME"),
    "heldPercentInstitutions":  ("% of shares held by institutions.", "POINT_IN_TIME"),

    # ── Analyst Estimates (POINT_IN_TIME) ──
    "targetHighPrice":          ("Analyst highest price target.", "POINT_IN_TIME"),
    "targetLowPrice":           ("Analyst lowest price target.", "POINT_IN_TIME"),
    "targetMeanPrice":          ("Analyst mean price target.", "POINT_IN_TIME"),
    "targetMedianPrice":        ("Analyst median price target.", "POINT_IN_TIME"),
    "recommendationMean":       ("Analyst consensus (1=Strong Buy … 5=Sell).", "POINT_IN_TIME"),
    "recommendationKey":        ("Analyst recommendation text (buy/hold/sell).", "POINT_IN_TIME"),
    "numberOfAnalystOpinions":  ("Number of analysts covering this stock.", "POINT_IN_TIME"),
    "averageAnalystRating":     ("Average analyst rating as string.", "POINT_IN_TIME"),

    # ── Pipeline-Derived (DERIVED — computed in training loop, time-aligned) ──
    "hist_momentum":        ("Annual return for the backtest year. Computed per-year, NO leakage.", "DERIVED"),
    "hist_volatility":      ("Annualized daily vol for the backtest year. Computed per-year, NO leakage.", "DERIVED"),
    "current_volatility":   ("Annualized vol from full daily returns. Used by Member B, not embedding.", "DERIVED"),
    "top_holdings":         ("ETF top holdings list (pipeline-generated string).", "METADATA"),

    # ── Identifiers (IDENTIFIER — not useful as ML features) ──
    "phone":        ("Company phone number.", "IDENTIFIER"),
    "fax":          ("Company fax number.", "IDENTIFIER"),
    "website":      ("Company website URL.", "IDENTIFIER"),
    "irWebsite":    ("Investor relations website URL.", "IDENTIFIER"),
    "shortName":    ("Short company name.", "IDENTIFIER"),
    "longName":     ("Full legal company name.", "IDENTIFIER"),
    "displayName":  ("Display name variant.", "IDENTIFIER"),
    "symbol":       ("Ticker symbol.", "IDENTIFIER"),
    "prevName":     ("Previous company name.", "IDENTIFIER"),

    # ── Free Text (TEXT — requires NLP, cannot one-hot encode) ──
    "longBusinessSummary":  ("Full company business description paragraph.", "TEXT"),
    "companyOfficers":      ("List of company officers (structured/nested).", "TEXT"),
    "executiveTeam":        ("Executive team information.", "TEXT"),

    # ── API/System Metadata (METADATA — no ML signal) ──
    "maxAge":                   ("Cache age in seconds (yfinance internal).", "METADATA"),
    "priceHint":                ("Decimal precision for price display.", "METADATA"),
    "currency":                 ("Trading currency (mostly USD).", "METADATA"),
    "financialCurrency":        ("Reporting currency.", "METADATA"),
    "language":                 ("Language code.", "METADATA"),
    "region":                   ("Region code.", "METADATA"),
    "typeDisp":                 ("Quote type display name.", "METADATA"),
    "quoteSourceName":          ("Data source name.", "METADATA"),
    "tradeable":                ("Whether tradeable on platform (boolean).", "METADATA"),
    "triggerable":              ("Whether alerts can be set (boolean).", "METADATA"),
    "cryptoTradeable":          ("Whether crypto trading available.", "METADATA"),
    "hasPrePostMarketData":     ("Whether pre/post market data exists.", "METADATA"),
    "customPriceAlertConfidence":("Alert confidence level.", "METADATA"),
    "corporateActions":         ("Corporate actions data.", "METADATA"),
    "messageBoardId":           ("Yahoo message board ID.", "METADATA"),
    "exchangeTimezoneName":     ("Exchange timezone name.", "METADATA"),
    "exchangeTimezoneShortName":("Exchange timezone abbreviation.", "METADATA"),
    "gmtOffSetMilliseconds":    ("GMT offset in milliseconds.", "METADATA"),
    "esgPopulated":             ("Whether ESG data is populated.", "METADATA"),
    "sourceInterval":           ("Data source polling interval.", "METADATA"),
    "exchangeDataDelayedBy":    ("Data delay in seconds.", "METADATA"),
    "firstTradeDateMilliseconds":("First trade date (epoch ms).", "METADATA"),
    "marketState":              ("Current market state (REGULAR, POST, PRE).", "METADATA"),
    "regularMarketTime":        ("Timestamp of last regular trade.", "METADATA"),
    "postMarketTime":           ("Timestamp of post-market data.", "METADATA"),
    "governanceEpochDate":      ("Date of governance assessment (epoch).", "METADATA"),
    "compensationAsOfEpochDate":("Date of compensation data (epoch).", "METADATA"),
    "exDividendDate":           ("Next ex-dividend date (epoch).", "METADATA"),
    "dividendDate":             ("Next dividend payment date (epoch).", "METADATA"),
    "lastDividendDate":         ("Date of most recent dividend (epoch).", "METADATA"),
    "lastSplitDate":            ("Date of last stock split (epoch).", "METADATA"),
    "sharesShortPreviousMonthDate": ("Date of prior month short data.", "METADATA"),
    "dateShortInterest":        ("Date of short interest data.", "METADATA"),
    "lastFiscalYearEnd":        ("Last fiscal year end (epoch).", "METADATA"),
    "nextFiscalYearEnd":        ("Next fiscal year end (epoch).", "METADATA"),
    "mostRecentQuarter":        ("Most recent quarter end (epoch).", "METADATA"),
    "earningsTimestamp":        ("Next earnings date (epoch).", "METADATA"),
    "earningsTimestampStart":   ("Earnings window start (epoch).", "METADATA"),
    "earningsTimestampEnd":     ("Earnings window end (epoch).", "METADATA"),
    "earningsCallTimestampStart":("Earnings call start (epoch).", "METADATA"),
    "earningsCallTimestampEnd": ("Earnings call end (epoch).", "METADATA"),
    "isEarningsDateEstimate":   ("Whether earnings date is estimated.", "METADATA"),
    "nameChangeDate":           ("Date of company name change.", "METADATA"),
    "ipoExpectedDate":          ("Expected IPO date.", "METADATA"),
    "prevExchange":             ("Previous exchange listing.", "METADATA"),
    "exchangeTransferDate":     ("Date of exchange transfer.", "METADATA"),
}

SAFETY_ICONS = {
    "SAFE_STATIC":   "🟢",
    "SLOW_CHANGING": "🟡",
    "POINT_IN_TIME": "🔴",
    "DERIVED":       "⚪",
    "METADATA":      "⬜",
    "TEXT":          "📝",
    "IDENTIFIER":   "🏷️",
    "UNKNOWN":       "❓",
}

def _resolve_column_info(col_name):
    """Look up a column in the registry, checking for one-hot prefixes if needed."""
    if col_name in COLUMN_SAFETY_REGISTRY:
        return COLUMN_SAFETY_REGISTRY[col_name]
    # Check if this is a one-hot encoded column (e.g. 'sector_Technology')
    for prefix in COLUMN_SAFETY_REGISTRY:
        if col_name.startswith(prefix + "_"):
            parent_desc, parent_safety = COLUMN_SAFETY_REGISTRY[prefix]
            suffix = col_name[len(prefix)+1:]
            return (f"One-hot from '{prefix}' = '{suffix}'", parent_safety)
    return ("(Not in registry — review manually)", "UNKNOWN")

def run_data_diagnostics(master_df, config):
    """Print comprehensive data dictionary, fill-rate analytics, and validate ML feature config."""
    feature_config = config.get("ml_training_features", [])
    
    # ── Section 1: Full Column Inventory ──
    print("\n" + "=" * 100)
    print("  DATA DICTIONARY — ALL COLUMNS IN master_df")
    print("  Total Columns:", len(master_df.columns), " | Total Assets:", len(master_df))
    print("=" * 100)
    print(f"{'#':<5} {'Column':<42} {'Dtype':<10} {'Non-Null':<9} {'Fill%':<7} {'Safety':<15} Description")
    print("-" * 150)
    
    safety_counts = {}
    for i, col in enumerate(master_df.columns):
        dtype_str = str(master_df[col].dtype)[:8]
        non_null = int(master_df[col].notna().sum())
        fill_pct = non_null / len(master_df) * 100 if len(master_df) > 0 else 0
        desc, safety = _resolve_column_info(col)
        icon = SAFETY_ICONS.get(safety, "❓")
        safety_counts[safety] = safety_counts.get(safety, 0) + 1
        
        in_model = " ◀ IN MODEL" if col in feature_config else ""
        fill_bar = "█" * int(fill_pct // 10) + "░" * (10 - int(fill_pct // 10))
        print(f"{i:<5} {col:<42} {dtype_str:<10} {non_null:<9} {fill_bar} {fill_pct:>5.1f}%  {icon} {safety:<13} {desc}{in_model}")
    
    # ── Section 2: Safety Class Summary ──
    print("\n" + "=" * 100)
    print("  SAFETY CLASS DISTRIBUTION")
    print("=" * 100)
    for cls in ["SAFE_STATIC", "SLOW_CHANGING", "POINT_IN_TIME", "DERIVED", "METADATA", "TEXT", "IDENTIFIER", "UNKNOWN"]:
        count = safety_counts.get(cls, 0)
        if count > 0:
            print(f"  {SAFETY_ICONS.get(cls, '❓')} {cls:<16} {count:>4} columns")
    
    # ── Section 3: ML Feature Validation ──
    print("\n" + "=" * 100)
    print("  ML FEATURE VALIDATION — Checking config['ml_training_features']")
    print("=" * 100)
    
    poisoning_warnings = []
    missing_warnings = []
    
    for f in feature_config:
        desc, safety = _resolve_column_info(f)
        icon = SAFETY_ICONS.get(safety, "❓")
        
        if safety == "DERIVED":
            print(f"  ✅ '{f}' — {icon} DERIVED: Computed per-year in training loop. No data leakage.")
            continue
        
        if f in master_df.columns:
            fill = master_df[f].notna().sum() / len(master_df) * 100 if len(master_df) > 0 else 0
            uniq = master_df[f].nunique()
            is_cat = master_df[f].dtype == 'object' or pd.api.types.is_categorical_dtype(master_df[f])
            type_info = f"Categorical ({uniq} unique → {uniq} one-hot cols)" if is_cat else "Numeric"
            
            if safety == "POINT_IN_TIME":
                print(f"  ⚠️  '{f}' — {icon} POINT_IN_TIME | Fill: {fill:.1f}% | {type_info}")
                print(f"       └─ WARNING: Uses TODAY's value for all backtest years → data leakage risk!")
                poisoning_warnings.append(f)
            elif safety == "SAFE_STATIC":
                print(f"  ✅ '{f}' — {icon} SAFE_STATIC | Fill: {fill:.1f}% | {type_info}")
            elif safety == "SLOW_CHANGING":
                print(f"  🟡 '{f}' — {icon} SLOW_CHANGING | Fill: {fill:.1f}% | {type_info}")
                print(f"       └─ Note: Today's snapshot used for all years. Low risk but not historically accurate.")
            elif safety in ["METADATA", "IDENTIFIER", "TEXT"]:
                print(f"  ❌ '{f}' — {icon} {safety} | This column type is not suitable for ML training.")
            else:
                print(f"  ❓ '{f}' — {icon} {safety} | Fill: {fill:.1f}% | {type_info} — Review manually.")
        else:
            # Check if it was already one-hot expanded (e.g. 'sector' → 'sector_Technology', ...)
            dummy_cols = [c for c in master_df.columns if c.startswith(f + "_")]
            if dummy_cols:
                print(f"  ✅ '{f}' — {icon} {safety} | Already one-hot expanded → {len(dummy_cols)} columns")
            else:
                print(f"  ❌ '{f}' — NOT FOUND in master_df and no one-hot expansion detected!")
                missing_warnings.append(f)
    
    # -- Section 3.5: Predicted Input Dimensionality --
    print("\n" + "=" * 100)
    print("  PREDICTED EMBEDDING INPUT VECTOR DIMENSIONALITY")
    print("=" * 100)
    print(f"  {'Feature':<35} {'Type':<15} {'Dims':>6}   Details")
    print("  " + "-" * 95)
    
    _total_pred = 0
    for f in feature_config:
        if f in ['hist_momentum', 'hist_volatility']:
            print(f"  {f:<35} {'DERIVED':<15} {1:>6}   Computed per-year in training loop")
            _total_pred += 1
        elif f in master_df.columns and pd.api.types.is_numeric_dtype(master_df[f]):
            print(f"  {f:<35} {'Numeric':<15} {1:>6}")
            _total_pred += 1
        else:
            _dc = [c for c in master_df.columns if c.startswith(f + '_')]
            if _dc:
                _sample = ', '.join(_dc[:3])
                _more = f" ... +{len(_dc)-3} more" if len(_dc) > 3 else ""
                print(f"  {f:<35} {'One-Hot':<15} {len(_dc):>6}   ({_sample}{_more})")
                _total_pred += len(_dc)
            elif f in master_df.columns:
                print(f"  {f:<35} {'Numeric':<15} {1:>6}")
                _total_pred += 1
            else:
                print(f"  {f:<35} {'NOT FOUND':<15} {'--':>6}   Will be skipped by training loop")
    
    print("  " + "-" * 95)
    print(f"  {'TOTAL PREDICTED INPUT DIMS':<35} {'':<15} {_total_pred:>6}")
    
    _hl = config.get('ml_hidden_layers', [32])
    _ed = config.get('ml_embedding_dim', 8)
    _oh = len(config.get('ml_target_horizons', [])) * 2
    print(f"\n  Network Architecture Preview:")
    _enc_p = [str(_total_pred)] + [f"{h}->ReLU" for h in _hl] + [f"{_ed}(embed)"]
    _dec_p = [f"{_ed}(embed)"] + [f"{h}->ReLU" for h in reversed(_hl)] + [f"{_oh}(output)"]
    print(f"  Encoder: {' -> '.join(_enc_p)}")
    print(f"  Decoder: {' -> '.join(_dec_p)}")
    _cr = _total_pred // _ed if _ed else 0
    print(f"  Compression Ratio: {_total_pred} -> {_ed} ({_cr}:1)")
    
    if _total_pred > 0 and _hl and _total_pred > _hl[0] * 3:
        print(f"\n  WARNING: Input dims ({_total_pred}) significantly larger than first hidden layer ({_hl[0]}).")
        print(f"      Consider increasing ml_hidden_layers[0] for better gradient flow.")
    
    # ── Section 4: Fill Rate Summary ──
    print("\n" + "=" * 100)
    print("  DATA QUALITY — FILL RATE ANALYTICS")
    print("=" * 100)
    
    fill_rates = master_df.notna().mean() * 100
    
    print(f"  Columns with 100% fill:   {(fill_rates == 100).sum():>4}")
    print(f"  Columns with 80-99% fill: {((fill_rates >= 80) & (fill_rates < 100)).sum():>4}")
    print(f"  Columns with 50-79% fill: {((fill_rates >= 50) & (fill_rates < 80)).sum():>4}")
    print(f"  Columns with 20-49% fill: {((fill_rates >= 20) & (fill_rates < 50)).sum():>4}")
    print(f"  Columns with  1-19% fill: {((fill_rates > 0) & (fill_rates < 20)).sum():>4}")
    print(f"  Columns with    0% fill:  {(fill_rates == 0).sum():>4}")
    
    empty_cols = fill_rates[fill_rates == 0].index.tolist()
    if empty_cols:
        print(f"\n  ⚠️  COMPLETELY EMPTY COLUMNS ({len(empty_cols)} total):")
        for c in empty_cols[:25]:
            print(f"      • {c}")
        if len(empty_cols) > 25:
            print(f"      ... and {len(empty_cols) - 25} more")
    
    sparse_cols = fill_rates[(fill_rates > 0) & (fill_rates < 20)].sort_values()
    if len(sparse_cols) > 0:
        print(f"\n  ⚠️  VERY SPARSE COLUMNS (<20% fill, {len(sparse_cols)} total):")
        for c, pct in sparse_cols.head(15).items():
            print(f"      • {c}: {pct:.1f}%")
        if len(sparse_cols) > 15:
            print(f"      ... and {len(sparse_cols) - 15} more")
    
    # ── Section 5: Final Verdict ──
    print("\n" + "=" * 100)
    print("  VALIDATION VERDICT")
    print("=" * 100)
    
    if poisoning_warnings:
        print(f"  🔴 DATA POISONING RISK: {len(poisoning_warnings)} feature(s) are POINT_IN_TIME:")
        for f in poisoning_warnings:
            print(f"     → '{f}' — today's value will be used for ALL historical backtest years")
        print(f"     Consider removing these or accepting the leakage trade-off.")
    else:
        print(f"  ✅ No POINT_IN_TIME features in config. Clean backtest.")
    
    if missing_warnings:
        print(f"  ❌ MISSING FEATURES: {len(missing_warnings)} feature(s) not found in master_df:")
        for f in missing_warnings:
            print(f"     → '{f}'")
    else:
        print(f"  ✅ All configured features exist in master_df.")
    
    print("=" * 100 + "\n")


In [ ]:
class AssetEmbeddingNet(nn.Module):
    def __init__(self, input_dim, embed_dim, output_dim, hidden_layers=None):
        super(AssetEmbeddingNet, self).__init__()
        if hidden_layers is None:
            hidden_layers = [32]
        
        # Encoder: progressively compress input -> embedding
        encoder_layers = []
        prev_dim = input_dim
        for h_dim in hidden_layers:
            encoder_layers.append(nn.Linear(prev_dim, h_dim))
            encoder_layers.append(nn.ReLU())
            prev_dim = h_dim
        encoder_layers.append(nn.Linear(prev_dim, embed_dim))
        self.encoder = nn.Sequential(*encoder_layers)
        
        # Decoder: mirror encoder (reversed hidden layers) -> output
        decoder_layers = []
        prev_dim = embed_dim
        for h_dim in reversed(hidden_layers):
            decoder_layers.append(nn.Linear(prev_dim, h_dim))
            decoder_layers.append(nn.ReLU())
            prev_dim = h_dim
        decoder_layers.append(nn.Linear(prev_dim, output_dim))
        self.decoder = nn.Sequential(*decoder_layers)
        
    def forward(self, x):
        emb = self.encoder(x)
        out = self.decoder(emb)
        return emb, out

def train_pytorch_embedding_model(master_df, price_matrix, daily_returns, config):
    print(f"[{time.strftime('%H:%M:%S')}] [Member A] Generating Historical Sample Paths...")
    
    # --- 1. SETUP TARGET HORIZONS AND ROLLING HISTORICAL METRICS ---
    horizons = config["ml_target_horizons"]
    annual_returns = daily_returns.resample('YE').apply(lambda x: (1+x).prod() - 1)
    annual_vols = daily_returns.resample('YE').apply(lambda x: x.std() * np.sqrt(252))
    
    feature_config = config.get("ml_training_features", [])
    
    # Scan processed master_df columns catching explicit numericals and one-hot prefixes
    model_input_cols = []
    for f in feature_config:
        if f in ['hist_momentum', 'hist_volatility']:
            continue
        if f in master_df.columns and pd.api.types.is_numeric_dtype(master_df[f]):
            model_input_cols.append(f)
        else:
            dummy_cols = [c for c in master_df.columns if c.startswith(f + "_")]
            model_input_cols.extend(dummy_cols)
            
    embed_features = [f for f in ["hist_momentum", "hist_volatility"] if f in feature_config] + model_input_cols
    print(f"\n[{time.strftime('%H:%M:%S')}] [Member A] Features used in Embedding Model:\n{embed_features}\n")
    # -- Per-feature dimensionality breakdown --
    print(f"[{time.strftime('%H:%M:%S')}] [Member A] Per-Feature Input Dimensionality:")
    print(f"  {'Feature':<35} {'Type':<12} {'Dims':>6}")
    print("  " + "-" * 55)
    _td = 0
    for _f in feature_config:
        if _f in ['hist_momentum', 'hist_volatility']:
            if _f in embed_features:
                print(f"  {_f:<35} {'DERIVED':<12} {1:>6}")
                _td += 1
        elif _f in master_df.columns and pd.api.types.is_numeric_dtype(master_df[_f]):
            print(f"  {_f:<35} {'Numeric':<12} {1:>6}")
            _td += 1
        else:
            _dc = [c for c in model_input_cols if c.startswith(_f + '_')]
            if _dc:
                print(f"  {_f:<35} {'One-Hot':<12} {len(_dc):>6}")
                _td += len(_dc)
    print("  " + "-" * 55)
    print(f"  {'TOTAL INPUT DIMENSIONS':<35} {'':<12} {_td:>6}\n")
    # --- 2. BACKWARDS TIME-SERIES GENERATION (Creating Historical Snapshots) ---
    X_list, Y_list = [], []
    valid_years = annual_returns.index.year
    
    for i in range(1, len(valid_years) - 1):
        for ticker in master_df.index:
            if ticker not in annual_returns.columns: continue
            
            hist_mom = annual_returns[ticker].iloc[i]
            hist_vol = annual_vols[ticker].iloc[i]
            if pd.isna(hist_mom) or pd.isna(hist_vol): continue
                
            # =========================================================
            # ⚠️ ML FEATURE SELECTION GATE (ANTI-DATA POISONING)
            # =========================================================
            # Selects solely from model_input_cols mapping back to Dashboard config bounds
            
            x_vec_list = []
            if "hist_momentum" in feature_config:
                x_vec_list.append(hist_mom)
            if "hist_volatility" in feature_config:
                x_vec_list.append(hist_vol)
                
            if len(model_input_cols) > 0:
                n_vec = master_df.loc[ticker, model_input_cols].values.astype(float)
                x_vec = np.concatenate([x_vec_list, n_vec])
            else:
                x_vec = np.array(x_vec_list)
            
            # --- 3. TARGET GENERATION & NaN SUPPLEMENTING ---
            y_vec = []
            valid_sample = False
            for h in horizons:
                if i + h < len(valid_years):
                    ret_target = annual_returns[ticker].iloc[i+1 : i+h+1].mean()
                    vol_target = annual_vols[ticker].iloc[i+1 : i+h+1].mean()
                    y_vec.extend([ret_target, vol_target])
                    if pd.notna(ret_target): valid_sample = True
                else:
                    y_vec.extend([np.nan, np.nan])
                    
            if valid_sample:
                X_list.append(x_vec)
                Y_list.append(y_vec)

    if len(X_list) == 0:
        print("[Member A] WARNING: Not enough historical data to generate PyTorch samples.")
        return {"master_df": master_df, "dynamic_embeddings": {t: np.random.normal(0,1,size=config["ml_embedding_dim"]) for t in master_df.index}, "price_matrix": price_matrix, "daily_returns": daily_returns, "mean_z_scores": None}
        
    X_tensor_raw = torch.tensor(np.array(X_list), dtype=torch.float32)
    Y_tensor_raw = torch.tensor(np.array(Y_list), dtype=torch.float32)
    
    # --- 4. DATA NORMALIZATION (Z-SCORES) ---
    print(f"[{time.strftime('%H:%M:%S')}] [Member A] Generated {len(X_list)} samples. Normalizing to Z-Scores via nanmean/nanstd...")
    Y_mean = torch.nanmean(Y_tensor_raw, dim=0)
    Y_std = torch.tensor(np.nanstd(Y_tensor_raw.numpy(), axis=0)) + 1e-8
    Y_std = torch.where(torch.isnan(Y_std), torch.ones_like(Y_std), Y_std) 
    Y_tensor = (Y_tensor_raw - Y_mean) / Y_std
    X_tensor = (X_tensor_raw - X_tensor_raw.mean(dim=0)) / (X_tensor_raw.std(dim=0) + 1e-8)
    
    dataset = TensorDataset(X_tensor, Y_tensor)
    loader = DataLoader(dataset, batch_size=config["ml_batch_size"], shuffle=True)
    
    # --- 5. INITIALIZE NEURAL NETWORK ---
    input_dim = X_tensor.shape[1]
    output_dim = len(horizons) * 2
    model = AssetEmbeddingNet(input_dim, config["ml_embedding_dim"], output_dim, config.get("ml_hidden_layers", [32]))
    
    # Print model architecture summary
    _hl = config.get("ml_hidden_layers", [32])
    _enc_parts = [str(input_dim)] + [f"{h} -> ReLU" for h in _hl] + [f"{config['ml_embedding_dim']} (embed)"]
    _dec_parts = [f"{config['ml_embedding_dim']} (embed)"] + [f"{h} -> ReLU" for h in reversed(_hl)] + [f"{output_dim} (output)"]
    print(f"\n[{time.strftime('%H:%M:%S')}] [Member A] Neural Network Architecture (Configurable):")
    print(f"  Encoder: {' -> '.join(_enc_parts)}")
    print(f"  Decoder: {' -> '.join(_dec_parts)}")
    _tp = sum(p.numel() for p in model.parameters())
    print(f"  Total Trainable Parameters: {_tp:,}")
    print(f"  Compression: {input_dim} dims -> {config['ml_embedding_dim']} dims ({input_dim // max(config['ml_embedding_dim'], 1)}:1)\n")
    
    optimizer = optim.Adam(model.parameters(), lr=config["ml_learning_rate"])
    
    weights_dict = config["ml_horizon_weights"]
    weights_array = []
    for h in horizons:
        w_h = weights_dict.get(h, 1.0)
        weights_array.extend([w_h, w_h]) 
    
    w_tensor = torch.tensor(weights_array, dtype=torch.float32)
    global_w_norm = w_tensor / w_tensor.sum() 
    
    print(f"[{time.strftime('%H:%M:%S')}] [Member A] Training PyTorch Model with Masked Weighted MSE...")
    for epoch in range(config["ml_epochs"]):
        model.train()
        total_loss = 0
        for bx, by in loader:
            optimizer.zero_grad()
            _, preds = model(bx)
            
            # --- 6. MASKED WEIGHTED MSE LOSS CALCULATION ---
            valid_mask = ~torch.isnan(by)
            batch_weights = global_w_norm.unsqueeze(0).expand_as(by)
            valid_weights = batch_weights * valid_mask.float()
            weight_sums = valid_weights.sum(dim=1, keepdim=True) + 1e-8
            normalized_active_weights = valid_weights / weight_sums
            err = (preds - by) ** 2
            clean_err = torch.nan_to_num(err, nan=0.0)
            weighted_err = clean_err * normalized_active_weights
            loss = weighted_err.sum(dim=1).mean()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            
        if epoch % 50 == 0 or epoch == config["ml_epochs"]-1:
            print(f"   Epoch {epoch}/{config['ml_epochs']} | Masked Weighted Loss: {total_loss/len(loader):.4f}")
            
    # --- 7. FREEZE MODEL WEIGHTS ---
    torch.save(model.state_dict(), "embedding_model.pth")
    print(f"[{time.strftime('%H:%M:%S')}] [Member A] Model FROZEN and saved to 'embedding_model.pth'.")
    
    # --- 8. GENERATE FINAL FEATURE EMBEDDING ---
    print(f"[{time.strftime('%H:%M:%S')}] [Member A] Extracting Current-Day Dynamic Vectors...")
    model.eval()
    dynamic_embeddings = {}
    with torch.no_grad():
        for ticker in master_df.index:
            if ticker not in annual_returns.columns:
                dynamic_embeddings[ticker] = np.zeros(config["ml_embedding_dim"])
                continue
                
            curr_mom = annual_returns[ticker].iloc[-1]
            curr_vol = annual_vols[ticker].iloc[-1]
            if pd.isna(curr_mom): curr_mom = 0.0
            if pd.isna(curr_vol): curr_vol = 0.0
            
            x_vec_list = []
            if "hist_momentum" in feature_config:
                x_vec_list.append(curr_mom)
            if "hist_volatility" in feature_config:
                x_vec_list.append(curr_vol)
                
            if len(model_input_cols) > 0:
                n_vec = master_df.loc[ticker, model_input_cols].values.astype(float)
                x_raw = np.concatenate([x_vec_list, n_vec])
            else:
                x_raw = np.array(x_vec_list)
            
            X_m = X_tensor_raw.mean(dim=0).numpy()
            X_s = X_tensor_raw.std(dim=0).numpy() + 1e-8
            x_norm = torch.tensor((x_raw - X_m) / X_s, dtype=torch.float32).unsqueeze(0)
            emb, _ = model(x_norm)
            dynamic_embeddings[ticker] = emb.squeeze(0).numpy()
            
    for ticker in master_df.index:
        if ticker in daily_returns.columns:
            master_df.loc[ticker, "current_volatility"] = daily_returns[ticker].std() * np.sqrt(252)
            
    return {"master_df": master_df, "dynamic_embeddings": dynamic_embeddings, "price_matrix": price_matrix, "daily_returns": daily_returns, "mean_z_scores": Y_mean.numpy()}


## [TEAM MEMBER B] Sector Scoring & Allocation Layer

In [ ]:
def cosine_similarity(v1, v2):
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-9)

def recommend_and_allocate_member_b(dataset, user_profile, user_dynamic_vector, theta=0.2, phi_weights=(1.0, 1.0)):
    w_sim, w_vol = phi_weights
    master_df = dataset["master_df"]
    
    scores = {}
    for ticker, asset_vector in dataset["dynamic_embeddings"].items():
        if ticker == "VBTIX": continue
        
        sim_score = cosine_similarity(user_dynamic_vector, asset_vector)
        volatility_penalty = master_df.loc[ticker, "current_volatility"] if "current_volatility" in master_df.columns and pd.notna(master_df.loc[ticker, "current_volatility"]) else 0.20
        
        scores[ticker] = (w_sim * sim_score) - (w_vol * volatility_penalty)
        
    filtered_assets = {t: s for t, s in scores.items() if s >= theta}
    if len(filtered_assets) == 0:
        filtered_assets = {k: v for k, v in sorted(scores.items(), key=lambda item: item[1], reverse=True)[:3]}
        print(f"[{time.strftime('%H:%M:%S')}] [Member B] WARNING: Threshold θ={theta} too high.")
        
    risk_user = user_profile['risk_tolerance']
    T = max(0.1, (11.0 - risk_user) / 2.0)
    
    exp_scores = {t: math.exp(s / T) for t, s in filtered_assets.items()}
    sum_exp = sum(exp_scores.values())
    
    return {"portfolio_weights": {t: exp_val / sum_exp for t, exp_val in exp_scores.items()}, "scores": filtered_assets}


## [TEAM MEMBER C] Evaluation Framework & Simulation

In [ ]:
def evaluate_portfolio_member_c(dataset, recommendations, user_profile):
    # RUNS LAYER 3 LOGICs
    weights = recommendations["portfolio_weights"]
    daily_returns = dataset["daily_returns"][list(weights.keys())]
    portfolio_returns = daily_returns.dot(pd.Series(weights))
    annual_returns = portfolio_returns.resample('YE').apply(lambda x: (1+x).prod() - 1)
    
    max_horizon_years = max(user_profile['goals'].keys())
    start_capital = user_profile['start_cap']
    successful_simulations, total_simulations = 0, 0
    terminal_values = []
    
    for start_year in range(len(annual_returns) - max_horizon_years + 1):
        if start_year + max_horizon_years > len(annual_returns): break
            
        current_balance = start_capital
        bankrupt = False
        
        path = annual_returns.iloc[start_year:start_year+max_horizon_years]
        for year_idx, yr_return in enumerate(path):
            current_balance = current_balance * (1 + yr_return)
            actual_year = year_idx + 1
            if actual_year in user_profile['goals']:
                withdrawal = user_profile['goals'][actual_year]
                current_balance -= withdrawal
                if current_balance < 0:
                    bankrupt = True
                    break
                    
        total_simulations += 1
        if not bankrupt:
            successful_simulations += 1
            terminal_values.append(current_balance)
            
    GFR = successful_simulations / total_simulations if total_simulations > 0 else 0
    ETV = np.median(terminal_values) if len(terminal_values) > 0 else 0
    alpha = 1.0
    objective_score = (alpha * ETV) if GFR >= 0.90 else (alpha * ETV) * (GFR / 0.90)
    
    return {"GFR": GFR, "ETV": ETV, "Objective_Function_Score": objective_score, "Total_Simulations": total_simulations}


## [UNIFIED DASHBOARD] Feedback Loop & Config Optimization

In [ ]:
# ==========================================
# 1. PARAMETERS & INITIALIZATION (ALL CONFIGS)
# ==========================================
PIPELINE_CONFIG = {
    # -----------------------
    # System Execution Limits
    # -----------------------
    "data_source_mode": "CSV", # Switch to 'MINE_AND_CACHE' to rebuild from Wikipedia
    "data_start_date": "2000-01-01",
    "scrape_delay": 0.3, # Seconds to legally delay HTTP drops
    
    # -----------------------
    # Column Extraction Scope
    # -----------------------
    # Dynamically reads columns from master_df into ML Tensor
    # Only SAFE_STATIC and DERIVED columns to prevent data poisoning in backtest
    "ml_training_features": [
        "hist_momentum", "hist_volatility",   # DERIVED - per-year, zero leakage
        "sector", "industry",                  # SAFE_STATIC - company classification
        "state", "quoteType", "exchange",      # SAFE_STATIC - geographic & market identity
        "fullTimeEmployees", "overallRisk",    # SLOW_CHANGING - stable fundamentals
    ],
    
    # -----------------------
    # Neural Network Geometry
    # -----------------------
    "ml_embedding_dim": 8,
    "ml_hidden_layers": [128, 64, 32],  # Configurable encoder/decoder hidden layer widths
    "ml_target_horizons": [1, 3, 5, 10, 15],
    "ml_horizon_weights": {1: 1.0, 3: 0.8, 5: 0.6, 10: 0.4, 15: 0.2}, 
    "ml_epochs": 150,
    "ml_batch_size": 64,
    "ml_learning_rate": 0.01,
}

USER_PROFILE = {
    "profile_name": "High-Stress Dual-Goal User",
    "risk_tolerance": 8.0, 
    "start_cap": 100000,
    "goals": {5: 40000, 15: 80000} 
}

# Generate User Matching Vector via dimensions parameter
USER_DYNAMIC_VECTOR = np.random.normal(0, 1, size=PIPELINE_CONFIG["ml_embedding_dim"]) 

print("="*80)
print("BOOTING MEMBER A PIPELINE")
print("="*80)

if PIPELINE_CONFIG["data_source_mode"] == "MINE_AND_CACHE":
    tickers_to_mine = fetch_sp1500_universe()
else:
    tickers_to_mine = []

master_df, price_matrix, daily_returns = generate_dataset_member_a(
    tickers=tickers_to_mine, 
    config=PIPELINE_CONFIG
)

# Run Data Dictionary & Validation Engine
run_data_diagnostics(master_df, PIPELINE_CONFIG)

DATA_CACHE = train_pytorch_embedding_model(master_df, price_matrix, daily_returns, PIPELINE_CONFIG)

# ==========================================
# 2. FEEDBACK LOOP: GRID SEARCH OPTIMIZATION
# ==========================================
print("\n" + "="*80)
print("RUNNING AUTOMATED THRESHOLD (θ) TUNING LOOP")
print("="*80)

threshold_grid = [-0.5, 0.0, 0.2, 0.5]
phi = (1.0, 0.5) 
results_log = []

for theta_val in threshold_grid:
    recs = recommend_and_allocate_member_b(
        dataset=DATA_CACHE, 
        user_profile=USER_PROFILE,
        user_dynamic_vector=USER_DYNAMIC_VECTOR,
        theta=theta_val,
        phi_weights=phi
    )
    metrics = evaluate_portfolio_member_c(DATA_CACHE, recs, USER_PROFILE)
    results_log.append({
        "Threshold (θ)": theta_val,
        "Assets Kept (K)": len(recs["portfolio_weights"]),
        "GFR (%)": f"{metrics['GFR']:.2%}",
        "ETV ($)": f"${metrics['ETV']:,.0f}",
        "Objective Score": float(metrics['Objective_Function_Score'])
    })

display(pd.DataFrame(results_log).sort_values(by="Objective Score", ascending=False))
